RAPID_Wapiti_project

Last edited 20260825 by Robby Jost

USE:
1. Initial cleaning and formatting of Master Sample Data worksheet.
2. Calculation of TSS parameters
3. Merging of field data with TSS, DOC, ions, and nutrients via the sample_num.


The output csv files are saved here: '/content/drive/Shareddrives/RAPID_Wapiti_project/water_samples_QC'

# Start up workflow:
load libraries -> connect to google drive -> load field data entry spreadsheet via url to the Google Sheet "wapiti_fire_water_sample_data" -> define functions.

# Cleaning operations:
remove extra columns, specify columns that are string, numeric, & datetime datatypes, declare no data values as -9999, format date & time columns.

# Dataset descriptions:
1. count and percentage of nodata (-9999) values in each column.
2. Field data rows grouped by site and sample date.

# Create parameter specific dataframes

# export dataframes
CURRENT_OUTPUTS:
1. *wapiti_water_data_entry_sample_data_CLEANED.csv*
  A csv file with cleaned original columns from field entry spreadsheet.
2.  *wapiti_water_TSS_sample_data_CLEANED.csv*
  A csv file with suspended sediment related columns.
3. *wapiti_water_DOC_sample_data_CLEANED.csv*
  A csv file with Dissolved organic carbon related columns
4. *wapiti_water_ALL_sample_data_CLEANED.csv*
  A csv file with newly derived TSS and DOC columns
# Inprogress
concentration data, calculated Detection Limits, and standard deviations.
Addition of Ions and Nurtients datasets when these get processsed.
---
### Developer notes on new features to add
Create a function that scans the output csv files and returns the most recent date of samples collected and the sample numbers.
Create another function that scans the Shimazu, nutrients (AA5600?) and IC folder in the Bergstrom_Lab drive scraping the latest file of processed data and the max sample number in the respective files.
Create a third function (or refactor the current merge function) to add the new batch of data to the ALL_CLEANED.csv file.


## **Import packages**

In [1]:
import datetime
import glob
import os
import gspread
import numpy as np
import pandas as pd
from google.colab import auth
from google.auth import default
from google.colab import drive

# Authenticate Colab and give access to your drive

In [2]:
# Running this cell will bring multiple prompts to access your drive. Accept and continue

# connect to your google drive (needed for saving files at the end)
drive.mount('/content/drive')
# Authenticate with Google Colab to access spreadsheets
auth.authenticate_user()
creds, _ = default()

gc = gspread.authorize(creds)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# out path to save files in wapiti shared drive
out_path = '/content/drive/Shareddrives/RAPID_Wapiti_project/water_samples_QC'

### Load Data from Google Sheet

In [4]:
# path to wapiti shared drive
root_path = '/content/drive/Shareddrives/RAPID_Wapiti_project'
# read in the spreadsheet
url = 'https://docs.google.com/spreadsheets/d/1bF-Za-wf4RdvGU84KIQ1ZHTNx46QItcQgyukLzkRKAM'
try:
    spreadsheet = gc.open_by_url(url)
    print(f'Successfully opened spreadsheet by url: {url}')
except gspread.exceptions.SpreadsheetNotFound:
    print(f"Spreadsheet '{url}' not found by url.")
    spreadsheet = None

Successfully opened spreadsheet by url: https://docs.google.com/spreadsheets/d/1bF-Za-wf4RdvGU84KIQ1ZHTNx46QItcQgyukLzkRKAM


In [5]:
if spreadsheet:
    # select the first worksheet "Master Sample Data"
    worksheet = spreadsheet.sheet1
    # Get all values from the worksheet and create a DataFrame
    records = worksheet.get_all_values()

    df = pd.DataFrame(records)

    print("Data loaded successfully into a pandas DataFrame.")
    #display(df.head())
else:
    print("Could not load data due to spreadsheet not being found or an invalid ID.")

Data loaded successfully into a pandas DataFrame.


## This section formats and cleans some artifacts in the data_entry spreadsheet that would otherwise mess with data analysis

In [6]:
header_row = df.iloc[0]
header_row = [val if val != '' else f"unamed_{i}" for i, val in enumerate(header_row)]
# set new headers and drop the first row
df.columns = header_row
df = df[1:].reset_index(drop=True)
df = df.replace(['-9999', -9999], np.nan)

In [7]:
# Identify all columns that contain 'Unnamed' (case-insensitive check)
unnamed_cols = [col for col in df.columns if 'unamed' in str(col).lower()]

# Drop the unnamed columns from your DataFrame
df.drop(columns=unnamed_cols, inplace=True)

In [8]:
# Define the columns you want to keep as strings/objects datatypes
exclude_cols = {'site','date', 'time','bottles_collected','tin_num', 'notes'}

for col in df.columns:
    if col not in exclude_cols:
        # Convert the rest to numeric, coercing errors to NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        # confirms the skipped columns that don't get converted
        print(f"Skipping numeric conversion for: {col}")



Skipping numeric conversion for: site
Skipping numeric conversion for: date
Skipping numeric conversion for: time
Skipping numeric conversion for: bottles_collected
Skipping numeric conversion for: tin_num
Skipping numeric conversion for: notes


In [9]:
# standardize datetime columns
# Convert Date (Standardizing to YYYY-MM-DD for Pandas internal use)
df['date'] = pd.to_datetime(df['date'], format='%m/%d/%y', errors='coerce')

# make sure everything is a string then remove colons and whitespace
df['time'] = df['time'].astype(str).str.replace(':', '').str.strip()


# re-insert the colon so it's "HH:MM"
df['time'] = df['time'].str.zfill(4) # Ensures "900" becomes "0900"
df['time'] = df['time'].str[:2] + ':' + df['time'].str[2:]

# Convert to a proper time object for calculations
df['time_cleaned'] = pd.to_datetime(df['time'], format='%H:%M', errors='coerce').dt.time

# Display to verify
print(df[['date', 'time', 'time_cleaned']].head())


        date   time time_cleaned
0 2024-10-11  15:00     15:00:00
1 2024-10-11  11:34     11:34:00
2 2024-10-11  14:15     14:15:00
3 2024-10-11  15:27     15:27:00
4 2024-10-12  10:20     10:20:00


# Calculate TSS columns

In [10]:

# columns to use for total suspended solids
tss_cols = [
    'sample_num', 'date','site', 'tin_num','tin_filter_weight', 'bottle_tare',
    'bottle_full', 'tin_filter_sed_weight', 'tin_filter_sed_ashed_weight'
]
tss_df = df[tss_cols].copy()

In [11]:
# convert from g to mg
tss_df['sed_weight_mg'] = (tss_df['tin_filter_sed_weight'] - tss_df['tin_filter_weight']) * 1000

tss_df['TSS_conc_mgL-1'] = (tss_df['bottle_full'] - tss_df['bottle_tare']) / 1000

tss_df["susp_solids_mg"] = (tss_df['tin_filter_sed_weight'] - tss_df['tin_filter_weight']) * 1000

tss_df["susp_sed_mg"] = (tss_df['tin_filter_sed_weight'] - tss_df['tin_filter_sed_ashed_weight']) * 1000

#  Shimadzu Datasets
1. Merge the processed TOC & NPOC csv files into a single dataframe.
2. Remove the 'blank', 'check1', 'check2' row. Are those important?
3. check for duplicates and remove


In [12]:
def clean_toc_data(file_path:str) -> pd.DataFrame:
    """
    Read all CSV files in a directory into a single DataFrame and clean it up.

    Args:
        file_path (str): The path to the directory containing the CSV files.

    Returns:
        pd.DataFrame: A DataFrame containing the cleaned TOC data.
    """

    # glob the spreadsheets
    csv_files = glob.glob(file_path)
    # read csv file into dataframe
    toc_processed_df = pd.concat([pd.read_csv(file, header=0) for file in csv_files], ignore_index=True)
    toc_processed_df.replace(['NA', -9999], np.nan, inplace=True)
    # copy the dataframe with the relevant columns
    toc_processed_cols = ['SampleNum', 'SiteName', 'TOC_conc', 'NPOC_conc']
    toc_df = toc_processed_df[toc_processed_cols].copy()
    # replace the column names to match the field entry spreadsheet
    toc_df.columns = ['sample_num', 'site', 'TOC_conc', 'NPOC_conc']
    # Drop the rows with NA values
    toc_df.dropna(axis=0, inplace=True)

    toc_df.sort_values(by='sample_num', inplace=True)

    # dropping the second duplicate - temporary measure
    toc_df_cleaned = toc_df.drop_duplicates(subset=['sample_num'], keep='first')

    return toc_df_cleaned

In [13]:
# file path to TOC data
toc_path = '/content/drive/Shareddrives/Watershed_Biogeochemistry_Lab/Projects/ShimadzuTOC/ProcessedData'
fname_string = '*_processedData*.csv'

In [14]:
if os.path.exists(toc_path):
  print(f"path found, cleaning TOC data")
  toc_df = clean_toc_data(os.path.join(toc_path, fname_string))
else:
  print(f"{toc_path} not found")

path found, cleaning TOC data


# Merge datasets

In [15]:
def merge_new_columns(
    primary_df: pd.DataFrame,
    incoming_df: pd.DataFrame,
    join_key: str = 'sample_num'
) -> pd.DataFrame:
    """Joins only the unique columns from an incoming DataFrame into a primary DataFrame.

    This function isolates columns in `incoming_df` that do not already exist in
    `primary_df`, filters out duplicate rows based on the `join_key` to prevent
    row multiplication, and performs a left merge. This prevents the creation
    of redundant column suffixes (e.g., '_x', '_y') for columns like site names
    that exist across multiple uncleaned datasets.

    Args:
        primary_df: The primary DataFrame acting as the left table.
            This dataframe retains all of its original columns and rows.
        incoming_df: The secondary DataFrame acting as the right table
            from which new, unique columns will be extracted.
        join_key: The column name to join the two dataframes on.
            Defaults to 'sample_num'.

    Returns:
        A new DataFrame containing all original data from `primary_df`
        augmented with the unique columns from `incoming_df`.

    Raises:
        KeyError: If the `join_key` is missing from either `primary_df` or
            `incoming_df`.
    """
    # 1. Get the sets of columns
    primary_cols = set(primary_df.columns)
    incoming_cols = set(incoming_df.columns)

    # 2. Find columns in incoming_df that are NOT in primary_df
    unique_new_cols = incoming_cols - primary_cols

    # 3. We MUST keep the join_key so we can actually perform the merge
    cols_to_keep = list(unique_new_cols) + [join_key]

    # 4. Filter the incoming dataframe to only those columns
    incoming_filtered = incoming_df[cols_to_keep]

    # 5. NEW: Clean up any duplicate join keys in the incoming data (our previous fix!)
    incoming_filtered = incoming_filtered.drop_duplicates(subset=[join_key], keep='first')

    # 6. Merge cleanly without worrying about _x or _y column suffixes
    refactored_df = pd.merge(primary_df, incoming_filtered, on=join_key, how='left')

    return refactored_df

In [16]:
# Start with your primary master df (df)
final_df = df.copy()

# Dynamically pull unique columns from tss_df
final_df = merge_new_columns(final_df, tss_df)

# Dynamically pull unique columns from toc_df
final_df = merge_new_columns(final_df, toc_df)

# guardrail to ensure no rows were accidentally multiplied
assert len(final_df) == len(df), f"Row count changed from {len(df)} to {len(final_df)}!"
print(f"Merge Complete! Final shape: {final_df.shape}. Original df shape: {df.shape}")

Merge Complete! Final shape: (616, 25). Original df shape: (616, 19)


# This section provides different descriptions of the data entry spreadsheet

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 616 entries, 0 to 615
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   sample_num                   616 non-null    int64         
 1   site                         616 non-null    object        
 2   date                         611 non-null    datetime64[ns]
 3   time                         616 non-null    object        
 4   field_temp                   560 non-null    float64       
 5   field_EC                     605 non-null    float64       
 6   lab_pH                       602 non-null    float64       
 7   lab_EC                       316 non-null    float64       
 8   bottles_collected            610 non-null    object        
 9   bottle_site_num              317 non-null    float64       
 10  tin_num                      602 non-null    object        
 11  tin_weight                   602 non-null    

In [18]:
# Calculate the count and percentage of missing data per column
missing_data = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    '% Missing': (df.isnull().sum() / len(df)) * 100
})

print(missing_data.sort_values(by='% Missing', ascending=False))

                             Missing Values  % Missing
lab_EC                                  300  48.701299
bottle_site_num                         299  48.538961
field_temp                               56   9.090909
tin_filter_sed_weight                    21   3.409091
bottle_full                              19   3.084416
tin_filter_sed_ashed_weight              19   3.084416
tin_weight                               14   2.272727
bottle_tare                              14   2.272727
tin_filter_weight                        14   2.272727
tin_num                                  14   2.272727
lab_pH                                   14   2.272727
field_EC                                 11   1.785714
bottles_collected                         6   0.974026
date                                      5   0.811688
time_cleaned                              5   0.811688
site                                      0   0.000000
sample_num                                0   0.000000
time      

## Groupby sample date

In [19]:
# see all the records for each date
for date, group in df.groupby("date"):
    print(f"--- Data for Site: {date} ---")
    display(group) # Shows the first 5 rows for each site
    print("\n")

--- Data for Site: 2024-10-11 00:00:00 ---


,sample_num,site,date,time,field_temp,field_EC,lab_pH,lab_EC,bottles_collected,bottle_site_num,tin_num,tin_weight,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,notes,time_cleaned
0,1,Warm_Sp,2024-10-11,15:00,NaN,NaN,8.39,NaN,"TSS, Nuts",NaN,19,1.075,1.208,88.57,1135.00,1.208,1.205,,15:00:00
1,2,SH_Cr,2024-10-11,11:34,NaN,NaN,7.91,NaN,"TSS, Nuts",NaN,26,1.083,1.214,88.54,1133.87,1.234,NaN,,11:34:00
2,3,Canyon_Cr_SFP,2024-10-11,14:15,NaN,NaN,7.97,NaN,"TSS, Nuts",NaN,25,1.098,1.232,87.50,1117.52,1.241,1.236,,14:15:00
3,4,BJ_Cr,2024-10-11,15:27,NaN,NaN,7.79,NaN,"TSS, Nuts",NaN,21,1.080,1.213,87.43,1080.30,1.238,NaN,,15:27:00




--- Data for Site: 2024-10-12 00:00:00 ---


,sample_num,site,date,time,field_temp,field_EC,lab_pH,lab_EC,bottles_collected,bottle_site_num,tin_num,tin_weight,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,notes,time_cleaned
4,5,SFP_Alder_Cr,2024-10-12,10:20,12.0,107.0,7.81,NaN,"TSS, Nuts, DOC, Alk",NaN,16,1.0950,1.228,88.35,1159.42,1.231,1.22800,,10:20:00
5,6,SFP_DW,2024-10-12,11:15,9.8,103.0,7.87,NaN,"TSS, Nuts, DOC, Alk",NaN,20,1.0600,1.195,88.49,1156.48,1.196,1.19400,,11:15:00
6,7,SFP_BJ,2024-10-12,12:30,9.4,98.4,NaN,NaN,DOC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Glass vial broke, sample gone",12:30:00
7,8,BJ_Cr,2024-10-12,12:30,8.0,54.0,NaN,NaN,DOC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,12:30:00
8,9,Warm_Sp,2024-10-12,13:15,13.0,114.0,NaN,NaN,DOC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,13:15:00
9,10,SFP_Chap,2024-10-12,13:30,10.5,89.0,NaN,NaN,DOC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Glass vial broke, sample gone",13:30:00
10,11,Chapman_Cr,2024-10-12,13:30,8.7,110.0,NaN,NaN,DOC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,13:30:00
11,12,SFP_Wolf_Cr,2024-10-12,14:45,15.4,84.2,8.22,NaN,"TSS, Nuts, DOC,Alk",NaN,18,1.1110,1.243,88.23,1153.16,1.244,1.24300,,14:45:00
12,13,Fox_Cr,2024-10-12,15:00,11.5,185.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,15:00:00
13,14,Canyon_Cr_SFP,2024-10-12,15:45,12.1,103.7,NaN,NaN,DOC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,15:45:00




--- Data for Site: 2024-10-13 00:00:00 ---


...

,sample_num,site,date,time,field_temp,field_EC,lab_pH,lab_EC,bottles_collected,bottle_site_num,tin_num,tin_weight,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,notes,time_cleaned
565,566,SFP_Wapiti,2026-07-10,08:10,10.1,41.9,7.48,51.34,"TSS, Nuts, DOC, Alk",12.0,RC15,1.1023,1.2354,87.88,1090.31,1.2379,1.2364,lab ec constant 0.947/cm,08:10:00
566,567,Bear_Cr,2026-07-10,08:20,9.4,78.6,7.29,84.71,"TSS, Nuts, DOC, Alk",19.0,32,1.0790,1.1959,87.47,1070.79,1.1984,1.1980,Whatman GF/F 934-AH used for TSS,08:20:00
567,568,SH_Cr,2026-07-10,08:35,10.9,55.3,7.26,60.33,"TSS, Nuts, DOC, Alk",8.0,41,1.0622,1.1952,88.54,1073.92,1.1985,1.1975,,08:35:00
568,569,Camp_Cr,2026-07-10,08:45,9.2,64.6,7.48,69.85,"TSS, Nuts, DOC, Alk",11.0,RC8,1.0730,1.2062,88.86,1102.53,1.2120,1.2105,,08:45:00
569,570,Canyon_Cr_SFP,2026-07-10,09:00,8.5,84.5,7.08,97.83,"TSS, Nuts, DOC, Alk",16.0,FC9,1.0899,1.2231,88.47,1131.77,1.2334,1.2315,,09:00:00
570,571,SFP_Canyon_Cr,2026-07-10,09:05,10.5,49.0,7.54,53.30,"TSS, Nuts, DOC, Alk",6.0,FC15,1.1078,1.2419,87.46,1138.26,1.2449,1.2439,,09:05:00
571,572,Fox_Cr,2026-07-10,09:25,11.3,178.3,7.24,192.10,"TSS, Nuts, DOC, Alk",2.0,FC29,0.9507,1.0847,89.62,1097.08,1.0885,1.0873,,09:25:00
572,573,SFP_Wolf_Cr,2026-07-10,09:40,10.7,59.7,7.45,64.87,"TSS, Nuts, DOC, Alk",20.0,FC18,1.0528,1.1851,88.29,1140.23,1.1890,1.1883,,09:40:00
573,574,HP,2026-07-10,10:15,8.5,80.4,7.37,86.55,"TSS, Nuts, DOC, Alk",15.0,59,1.0921,1.2242,87.99,1126.26,1.2276,1.2264,,10:15:00
574,575,Avalanche_Cr,2026-07-10,10:35,10.4,105.8,7.43,115.50,"TSS, Nuts, DOC, Alk",4.0,4,1.0820,1.2142,87.91,1117.66,1.2158,1.2151,,10:35:00




--- Data for Site: 2026-07-31 00:00:00 ---


,sample_num,site,date,time,field_temp,field_EC,lab_pH,lab_EC,bottles_collected,bottle_site_num,tin_num,tin_weight,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,notes,time_cleaned
582,583,SFP_Wapiti,2026-07-31,08:50,11.4,46.1,6.85,66.04,"TSS, Nuts, Alk",15.0,HZ,1.0700,1.1874,87.95,1124.17,1.3474,1.3412,non-regular sampling following 07/22 debris fl...,08:50:00
583,584,Bear_Cr,2026-07-31,09:00,10.6,88.6,7.15,92.61,"TSS, Nuts, Alk",12.0,FC20,1.1183,1.2352,87.89,1111.53,1.3253,1.3233,Whatman GF/F 934-AH used for TSS,09:00:00
584,585,SH_Cr,2026-07-31,09:40,12.5,54.9,6.92,60.65,"TSS, Nuts, DOC, Alk",17.0,FC14,1.0819,1.1995,88.33,1081.08,1.2013,1.2005,Whatman GF/F 934-AH used for DOC/TSS,09:40:00
585,586,Camp_Cr,2026-07-31,09:55,10.6,67.1,6.97,70.39,"TSS, Nuts, DOC, Alk",4.0,27,1.1011,1.2195,87.91,1112.95,1.2223,1.2216,Whatman GF/F 934-AH used for DOC/TSS,09:55:00
586,587,Canyon_Cr_SFP,2026-07-31,10:10,9.6,98.1,5.90,102.90,"TSS, Nuts, DOC, Alk",1.0,47,1.0971,1.2147,88.38,1128.62,1.3356,1.3293,Whatman GF/F 934-AH used for DOC/TSS,10:10:00
587,588,SFP_Canyon_Cr,2026-07-31,10:15,12.3,66.0,6.90,69.41,"TSS, Nuts, Alk",8.0,34,1.0656,1.1827,88.53,1116.02,2.0069,1.9983,Whatman GF/F 934-AH used for TSS,10:15:00
588,589,Fox_Cr,2026-07-31,10:30,13.2,186.2,6.40,194.80,"TSS, Nuts, Alk",9.0,37,1.0813,1.1982,88.35,1092.37,1.1996,1.1990,Whatman GF/F 934-AH used for TSS,10:30:00
589,590,SFP_Wolf_Cr,2026-07-31,10:40,12.5,76.2,6.70,80.94,"TSS, Nuts, DOC, Alk",6.0,6,1.1045,1.2221,87.43,1126.54,1.6601,1.6520,Whatman GF/F 934-AH used for DOC/TSS,10:40:00
590,591,HP,2026-07-31,11:10,11.0,90.1,6.18,94.80,"TSS, Nuts, Alk",18.0,51,1.0628,1.1792,87.84,1132.66,1.3256,1.3218,Whatman GF/F 934-AH used for TSS,11:10:00
591,592,Avalanche_Cr,2026-07-31,11:30,12.6,114.7,6.83,120.00,"TSS, Nuts, Alk",20.0,16,1.0968,1.2136,88.27,1140.21,1.2139,1.2137,Whatman GF/F 934-AH used for TSS,11:30:00




--- Data for Site: 2026-08-06 00:00:00 ---


,sample_num,site,date,time,field_temp,field_EC,lab_pH,lab_EC,bottles_collected,bottle_site_num,tin_num,tin_weight,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,notes,time_cleaned
599,600,SFP_Wapiti,2026-08-06,07:50,10.2,66.1,7.44,70.10,"TSS, Nuts, Alk",7.0,25,1.0984,1.2248,89.01,1074.28,1.3017,1.3002,duplicate TSS bottle labled site 7. lab EC cel...,07:50:00
600,601,Bear_Cr,2026-08-06,08:05,8.7,87.5,7.42,95.30,"TSS, Nuts, Alk",18.0,FC12,1.0886,1.2146,87.84,1130.70,1.2367,1.2355,duplicate TSS bottle labled site 18,08:05:00
601,602,SH_Cr,2026-08-06,08:20,10.6,58.1,7.35,61.72,"TSS, Nuts, Alk",13.0,FC17,1.0896,1.2162,88.73,1100.74,1.2174,1.2170,duplicate TSS bottle labled site 13,08:20:00
602,603,Camp_Cr,2026-08-06,08:30,9.2,67.8,7.31,71.39,"TSS, Nuts, Alk",10.0,FC26,1.0796,1.2072,87.53,1112.36,1.2102,1.2093,no duplicate TSS,08:30:00
603,604,Canyon_Cr_SFP,2026-08-06,08:45,8.2,93.0,7.33,106.90,"TSS, Nuts, Alk",6.0,2,1.0979,1.2243,87.45,1060.33,1.2941,1.2907,duplicate TSS bottle labeled site 6,08:45:00
604,605,SFP_Canyon_Cr,2026-08-06,08:50,10.1,67.8,7.14,75.31,"TSS, Nuts, Alk",2.0,38,1.0985,1.2257,89.61,1137.21,1.4605,1.4571,duplicate TSS bottle labeled site 2,08:50:00
605,606,Fox_Cr,2026-08-06,09:10,11.1,180.1,7.35,196.20,"TSS, Nuts, Alk",17.0,RC8,1.0725,1.1983,88.33,1099.74,1.1994,1.1986,no duplicate TSS,09:10:00
606,607,SFP_Wolf_Cr,2026-08-06,09:20,10.1,80.1,7.04,85.49,"TSS, Nuts, Alk",1.0,32,1.0791,1.2056,88.34,1136.23,1.3860,1.3829,no duplicate TSS,09:20:00
607,608,HP,2026-08-06,09:50,8.2,92.4,7.11,96.88,"TSS, Nuts, Alk",11.0,41,1.0625,1.1886,88.87,1133.05,1.2093,1.2081,no duplicate TSS,09:50:00
608,609,Avalanche_Cr,2026-08-06,10:05,10.2,116.6,6.84,122.90,"TSS, Nuts, Alk",4.0,4,1.0826,1.2096,87.90,1104.40,1.2106,1.2099,no duplicate TSS,10:05:00


## Group by site

In [20]:
for site, group in df.groupby("site"):
    print(f"--- Data for Site: {site} ---")
    display(group)
    print("\n")

--- Data for Site: Avalanche_Cr ---


,sample_num,site,date,time,field_temp,field_EC,lab_pH,lab_EC,bottles_collected,bottle_site_num,tin_num,tin_weight,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,notes,time_cleaned
30,31,Avalanche_Cr,2024-10-19,17:40,9.60,130.5,8.13,NaN,"TSS, Nuts, DOC, Alk",NaN,4,1.10000,1.23000,101.45,1110.23,1.24000,1.23700,,17:40:00
37,38,Avalanche_Cr,2024-11-03,13:15,4.00,123.2,8.02,NaN,"TSS, Nuts, DOC, Alk",NaN,31,1.09400,1.21700,87.64,1123.76,1.26950,1.26391,,13:15:00
57,58,Avalanche_Cr,2024-11-09,11:15,NaN,127.5,7.85,NaN,"TSS, Nuts, DOC, Alk",NaN,51,1.06200,1.18500,88.77,1125.45,1.18642,1.18574,,11:15:00
74,75,Avalanche_Cr,2024-11-24,11:55,NaN,126.9,8.11,NaN,"TSS, Nuts, DOC, Alk",NaN,16,1.09612,1.21856,88.49,1124.38,1.22548,1.22249,,11:55:00
89,90,Avalanche_Cr,2025-01-17,11:45,1.70,116.8,7.38,NaN,"TSS, Nuts, DOC, Alk",NaN,51,1.06185,1.19312,88.34,1063.49,1.19561,1.19490,Lab(omega) EC = 145.3 @7.2C,11:45:00
104,105,Avalanche_Cr,2025-01-30,09:08,0.60,124.3,7.75,NaN,"TSS, Nuts, DOC, Alk",NaN,22,1.11063,1.24091,88.43,644.80,1.24326,1.24241,Lab(metrohm) EC = 127.8 @5.9C,09:08:00
119,120,Avalanche_Cr,NaT,0n:an,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Road Closed, could not sample",NaT
135,136,Avalanche_Cr,2025-03-02,11:26,3.20,115.3,7.60,NaN,"TSS, Nuts, DOC, Alk",NaN,5,1.08899,1.21565,88.71,1142.03,1.27106,1.26811,,11:26:00
182,183,Avalanche_Cr,2025-04-15,13:45,5.80,74.8,7.55,NaN,"TSS, Nuts, DOC, Alk",NaN,25,1.09805,1.22438,88.55,1152.61,1.43208,1.33420,,13:45:00
200,201,Avalanche_Cr,2025-04-28,11:25,5.70,64.2,7.77,NaN,"TSS, Nuts, DOC, Alk",NaN,8,1.10662,1.23833,88.46,1143.78,1.76401,1.74821,relatively large sandy grains,11:25:00




--- Data for Site: BJ_Cr ---


,sample_num,site,date,time,field_temp,field_EC,lab_pH,lab_EC,bottles_collected,bottle_site_num,tin_num,tin_weight,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,notes,time_cleaned
3,4,BJ_Cr,2024-10-11,15:27,NaN,NaN,7.79,NaN,"TSS, Nuts",NaN,21,1.08000,1.213000,87.430,1080.30,1.23800,NaN,,15:27:00
7,8,BJ_Cr,2024-10-12,12:30,8.00,54.0,NaN,NaN,DOC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,12:30:00
20,21,BJ_Cr,2024-10-19,11:00,4.20,54.1,7.87,NaN,"TSS, Nuts, DOC, Alk",NaN,3,1.09000,1.230000,101.060,1082.96,1.24600,1.23700,"Glass vial broke, sample gone",11:00:00
48,49,BJ_Cr,2024-11-04,14:20,4.30,52.3,8.01,NaN,"TSS, Nuts, DOC, Alk",NaN,41,1.06200,1.186000,88.580,1075.83,1.19220,1.18744,,14:20:00
52,53,BJ_Cr,2024-11-09,10:15,NaN,61.6,8.12,NaN,"TSS, Nuts, DOC, Alk",NaN,46,1.20100,1.223000,88.560,1119.97,1.22534,1.22395,,10:15:00
69,70,BJ_Cr,2024-11-24,10:53,NaN,70.3,7.95,NaN,"TSS, Nuts, DOC, Alk",NaN,5,1.08887,1.211970,88.250,1130.70,1.21413,1.21233,,10:53:00
84,85,BJ_Cr,2025-01-17,09:13,1.10,52.2,7.21,NaN,"TSS, Nuts, DOC, Alk",NaN,9,1.06293,1.189689,87.460,1034.02,1.18888,1.18744,Lab(omega) EC = 64.67 @7.8C,09:13:00
99,100,BJ_Cr,2025-01-30,15:00,0.80,49.7,7.95,NaN,"TSS, Nuts, DOC, Alk",NaN,20,1.06124,1.192530,88.370,511.10,1.19471,1.19341,"Had to break through ice for sample,Lab(metroh...",15:00:00
114,115,BJ_Cr,2025-02-15,14:00,2.20,64.2,7.99,NaN,"TSS, Nuts, DOC, Alk",NaN,51,1.06219,1.193690,88.420,1092.78,1.19824,1.19601,Had to break through ice,14:00:00
130,131,BJ_Cr,2025-03-02,10:15,1.80,55.1,7.35,NaN,"TSS, Nuts, DOC, Alk",NaN,25,1.09785,1.223710,88.380,1107.88,1.22524,1.22442,,10:15:00




--- Data for Site: Bear_Cr ---


...

,sample_num,site,date,time,field_temp,field_EC,lab_pH,lab_EC,bottles_collected,bottle_site_num,tin_num,tin_weight,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,notes,time_cleaned
1,2,SH_Cr,2024-10-11,11:34,NaN,NaN,7.91,NaN,"TSS, Nuts",NaN,26,1.08300,1.21400,88.540,1133.87,1.23400,NaN,,11:34:00
15,16,SH_Cr,2024-10-12,16:30,13.00,98.3,8.05,NaN,"TSS, Nuts, DOC, Alk",NaN,23,1.11030,1.23700,87.630,1147.85,1.24000,1.23791,,16:30:00
16,17,SH_Cr,2024-10-13,16:00,13.30,100.2,7.57,NaN,"TSS, Nuts, DOC, Alk",NaN,24,1.06500,1.19900,88.810,1141.80,1.21800,NaN,"Glass vial broke, sample gone",16:00:00
28,29,SH_Cr,2024-10-19,14:10,8.10,89.5,7.90,NaN,"TSS, Nuts, DOC, Alk",NaN,12,1.06700,1.19600,101.430,1001.98,1.21400,1.21100,"Glass vial broke, sample gone",14:10:00
35,36,SH_Cr,2024-11-03,11:10,1.70,82.1,7.92,NaN,"TSS, Nuts, DOC, Alk",NaN,29,1.11700,1.24300,88.790,1120.02,1.25460,1.24751,,11:10:00
64,65,SH_Cr,2024-11-09,12:30,NaN,80.5,8.02,NaN,"TSS, Nuts, DOC, Alk",NaN,57,1.07400,1.20400,88.590,1067.57,1.20766,1.20682,,12:30:00
81,82,SH_Cr,2024-11-24,15:11,NaN,79.4,8.10,NaN,"TSS, Nuts, DOC, Alk",NaN,9,1.06299,1.18718,887.590,1126.84,1.23012,1.22715,,15:11:00
96,97,SH_Cr,2025-01-17,13:41,1.30,65.0,7.37,NaN,"TSS, Nuts, DOC, Alk",NaN,8,1.10626,1.23723,88.230,1053.98,1.41141,1.40853,"Lab(metrohm) EC = 81.43 @8.6C, **used both met...",13:41:00
111,112,SH_Cr,2025-01-30,11:22,0.30,20.9,7.96,NaN,"TSS, Nuts, DOC, Alk",NaN,5,1.08864,1.21926,88.220,1124.90,1.23974,1.23806,"Had to break through ice for sample,Lab(metroh...",11:22:00
126,127,SH_Cr,2025-02-15,14:20,1.50,67.9,NaN,NaN,"TSS, Nuts, DOC, Alk",NaN,46,1.10091,1.23171,88.110,737.44,1.25185,1.25060,"1L Bottle not all the way full, had to break t...",14:20:00




--- Data for Site: Wapiti_Cr ---


,sample_num,site,date,time,field_temp,field_EC,lab_pH,lab_EC,bottles_collected,bottle_site_num,tin_num,tin_weight,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,notes,time_cleaned
66,67,Wapiti_Cr,2024-11-09,16:00,NaN,87.5,7.92,NaN,"TSS, Nuts, DOC, Alk",NaN,59,1.09200,1.21300,88.08,1034.73,1.21914,1.21791,,16:00:00
185,186,Wapiti_Cr,2025-04-15,10:20,3.10,62.3,7.70,NaN,"TSS, Nuts, DOC, Alk",NaN,9,1.06343,1.18968,88.70,1145.44,1.19419,1.19271,,10:20:00
203,204,Wapiti_Cr,2025-04-28,12:20,5.40,58.8,7.84,NaN,"TSS, Nuts, DOC, Alk",NaN,11,1.08554,1.21706,87.63,1101.13,1.22485,1.22263,,12:20:00
220,221,Wapiti_Cr,2025-05-10,10:05,5.70,56.8,7.13,NaN,"TSS, Nuts, DOC, Alk",NaN,8,1.10724,1.24071,89.64,1149.96,1.26535,1.25607,,10:05:00
239,240,Wapiti_Cr,2025-05-27,12:50,7.80,49.5,7.37,NaN,"TSS, Nuts, DOC, Alk",NaN,15,1.08535,1.21751,88.51,1131.93,1.23264,1.22865,,12:50:00
258,259,Wapiti_Cr,2025-06-08,08:50,6.40,10.8,7.35,NaN,"TSS, Nuts, DOC, Alk",NaN,50,1.08794,1.21544,88.59,1149.98,1.22131,1.21936,,08:50:00
277,278,Wapiti_Cr,2025-06-22,08:50,5.10,73.7,7.43,NaN,"TSS, Nuts, DOC, Alk",NaN,15,1.08527,1.21230,87.95,1153.59,1.21647,1.21513,Rained/snowed late 6/21 through 13:30 on 6/22,08:50:00
297,298,Wapiti_Cr,2025-07-07,11:15,8.60,130.5,7.79,61.19,"TSS, Nuts, DOC, Alk",NaN,15,1.08534,1.21122,88.25,1020.30,1.21546,1.21391,Large rainstorm 7/4,11:15:00
316,317,Wapiti_Cr,2025-07-21,09:04,7.08,76.0,7.39,NaN,"TSS, Nuts, DOC, Alk",15.0,14,1.07708,1.20592,89.06,1000.35,1.21112,1.20934,used old YSI probe for temp and us,09:04:00
335,336,Wapiti_Cr,2025-08-01,10:00,7.69,79.0,7.98,83.49,"TSS, Nuts, DOC, Alk",19.0,5,1.08900,1.22100,89.61,1061.07,1.22200,1.22100,rainstorm night of 7/31 before sampling,10:00:00




--- Data for Site: Warm_Sp ---


,sample_num,site,date,time,field_temp,field_EC,lab_pH,lab_EC,bottles_collected,bottle_site_num,tin_num,tin_weight,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,notes,time_cleaned
0,1,Warm_Sp,2024-10-11,15:00,NaN,NaN,8.39,NaN,"TSS, Nuts",NaN,19,1.07500,1.20800,88.570,1135.00,1.20800,1.20500,,15:00:00
8,9,Warm_Sp,2024-10-12,13:15,13.00,114.0,NaN,NaN,DOC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,13:15:00
21,22,Warm_Sp,2024-10-19,11:40,8.20,115.2,8.27,NaN,"TSS, Nuts, DOC, Alk",NaN,7,1.08600,1.21700,100.990,1135.94,1.21400,1.21200,"Glass vial broke, sample gone",11:40:00
46,47,Warm_Sp,2024-11-04,13:40,8.20,109.4,8.14,NaN,"TSS, Nuts, DOC, Alk",NaN,39,1.10100,1.22900,87.870,1120.70,1.23600,1.23234,,13:40:00
54,55,Warm_Sp,2024-11-09,10:35,NaN,116.1,8.02,NaN,"TSS, Nuts, DOC, Alk",NaN,48,1.09800,1.22200,88.590,1122.91,1.22274,1.22237,,10:35:00
71,72,Warm_Sp,2024-11-24,11:07,NaN,112.4,8.01,NaN,"TSS, Nuts, DOC, Alk",NaN,14,1.07599,1.19937,88.320,1138.31,1.24092,1.23873,,11:07:00
86,87,Warm_Sp,2025-01-17,10:35,5.30,107.7,7.44,NaN,"TSS, Nuts, DOC, Alk",NaN,46,1.10076,1.23195,87.440,1123.14,1.23693,1.23622,Lab(omega) EC = 133.9 @8.0C,10:35:00
101,102,Warm_Sp,2025-01-30,14:06,6.30,98.7,7.97,NaN,"TSS, Nuts, DOC, Alk",NaN,8,1.10623,1.23845,88.330,1168.10,1.24267,1.24186,Lab(metrohm) EC = 125.7 @6.4C,14:06:00
116,117,Warm_Sp,2025-02-15,15:40,8.40,120.6,7.94,NaN,"TSS, Nuts, DOC, Alk",NaN,50,1.08724,1.21766,88.710,1123.34,1.22059,1.21977,,15:40:00
132,133,Warm_Sp,2025-03-02,18:30,9.10,114.3,7.51,NaN,"TSS, Nuts, DOC, Alk",NaN,22,1.11082,1.23651,88.400,1094.01,1.26331,1.26164,,18:30:00


## TSS data descriptions

In [21]:
for site, group in tss_df.groupby("site"):
    print(f"--- TSS Data for Site: {site} ---")
    display(group)
    print("\n")

--- TSS Data for Site: Avalanche_Cr ---


,sample_num,date,site,tin_num,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,sed_weight_mg,TSS_conc_mgL-1,susp_solids_mg,susp_sed_mg
30,31,2024-10-19,Avalanche_Cr,4,1.23000,101.45,1110.23,1.24000,1.23700,10.00,1.00878,10.00,3.00
37,38,2024-11-03,Avalanche_Cr,31,1.21700,87.64,1123.76,1.26950,1.26391,52.50,1.03612,52.50,5.59
57,58,2024-11-09,Avalanche_Cr,51,1.18500,88.77,1125.45,1.18642,1.18574,1.42,1.03668,1.42,0.68
74,75,2024-11-24,Avalanche_Cr,16,1.21856,88.49,1124.38,1.22548,1.22249,6.92,1.03589,6.92,2.99
89,90,2025-01-17,Avalanche_Cr,51,1.19312,88.34,1063.49,1.19561,1.19490,2.49,0.97515,2.49,0.71
104,105,2025-01-30,Avalanche_Cr,22,1.24091,88.43,644.80,1.24326,1.24241,2.35,0.55637,2.35,0.85
119,120,NaT,Avalanche_Cr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
135,136,2025-03-02,Avalanche_Cr,5,1.21565,88.71,1142.03,1.27106,1.26811,55.41,1.05332,55.41,2.95
182,183,2025-04-15,Avalanche_Cr,25,1.22438,88.55,1152.61,1.43208,1.33420,207.70,1.06406,207.70,97.88
200,201,2025-04-28,Avalanche_Cr,8,1.23833,88.46,1143.78,1.76401,1.74821,525.68,1.05532,525.68,15.80




--- TSS Data for Site: BJ_Cr ---


,sample_num,date,site,tin_num,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,sed_weight_mg,TSS_conc_mgL-1,susp_solids_mg,susp_sed_mg
3,4,2024-10-11,BJ_Cr,21,1.213000,87.430,1080.30,1.23800,NaN,25.000,0.992870,25.000,NaN
7,8,2024-10-12,BJ_Cr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,21,2024-10-19,BJ_Cr,3,1.230000,101.060,1082.96,1.24600,1.23700,16.000,0.981900,16.000,9.00
48,49,2024-11-04,BJ_Cr,41,1.186000,88.580,1075.83,1.19220,1.18744,6.200,0.987250,6.200,4.76
52,53,2024-11-09,BJ_Cr,46,1.223000,88.560,1119.97,1.22534,1.22395,2.340,1.031410,2.340,1.39
69,70,2024-11-24,BJ_Cr,5,1.211970,88.250,1130.70,1.21413,1.21233,2.160,1.042450,2.160,1.80
84,85,2025-01-17,BJ_Cr,9,1.189689,87.460,1034.02,1.18888,1.18744,-0.809,0.946560,-0.809,1.44
99,100,2025-01-30,BJ_Cr,20,1.192530,88.370,511.10,1.19471,1.19341,2.180,0.422730,2.180,1.30
114,115,2025-02-15,BJ_Cr,51,1.193690,88.420,1092.78,1.19824,1.19601,4.550,1.004360,4.550,2.23
130,131,2025-03-02,BJ_Cr,25,1.223710,88.380,1107.88,1.22524,1.22442,1.530,1.019500,1.530,0.82




--- TSS Data for Site: Bear_Cr ---


...

,sample_num,date,site,tin_num,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,sed_weight_mg,TSS_conc_mgL-1,susp_solids_mg,susp_sed_mg
1,2,2024-10-11,SH_Cr,26,1.21400,88.540,1133.87,1.23400,NaN,20.00,1.045330,20.00,NaN
15,16,2024-10-12,SH_Cr,23,1.23700,87.630,1147.85,1.24000,1.23791,3.00,1.060220,3.00,2.09
16,17,2024-10-13,SH_Cr,24,1.19900,88.810,1141.80,1.21800,NaN,19.00,1.052990,19.00,NaN
28,29,2024-10-19,SH_Cr,12,1.19600,101.430,1001.98,1.21400,1.21100,18.00,0.900550,18.00,3.00
35,36,2024-11-03,SH_Cr,29,1.24300,88.790,1120.02,1.25460,1.24751,11.60,1.031230,11.60,7.09
64,65,2024-11-09,SH_Cr,57,1.20400,88.590,1067.57,1.20766,1.20682,3.66,0.978980,3.66,0.84
81,82,2024-11-24,SH_Cr,9,1.18718,887.590,1126.84,1.23012,1.22715,42.94,0.239250,42.94,2.97
96,97,2025-01-17,SH_Cr,8,1.23723,88.230,1053.98,1.41141,1.40853,174.18,0.965750,174.18,2.88
111,112,2025-01-30,SH_Cr,5,1.21926,88.220,1124.90,1.23974,1.23806,20.48,1.036680,20.48,1.68
126,127,2025-02-15,SH_Cr,46,1.23171,88.110,737.44,1.25185,1.25060,20.14,0.649330,20.14,1.25




--- TSS Data for Site: Wapiti_Cr ---


,sample_num,date,site,tin_num,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,sed_weight_mg,TSS_conc_mgL-1,susp_solids_mg,susp_sed_mg
66,67,2024-11-09,Wapiti_Cr,59,1.21300,88.08,1034.73,1.21914,1.21791,6.14,0.94665,6.14,1.23
185,186,2025-04-15,Wapiti_Cr,9,1.18968,88.70,1145.44,1.19419,1.19271,4.51,1.05674,4.51,1.48
203,204,2025-04-28,Wapiti_Cr,11,1.21706,87.63,1101.13,1.22485,1.22263,7.79,1.01350,7.79,2.22
220,221,2025-05-10,Wapiti_Cr,8,1.24071,89.64,1149.96,1.26535,1.25607,24.64,1.06032,24.64,9.28
239,240,2025-05-27,Wapiti_Cr,15,1.21751,88.51,1131.93,1.23264,1.22865,15.13,1.04342,15.13,3.99
258,259,2025-06-08,Wapiti_Cr,50,1.21544,88.59,1149.98,1.22131,1.21936,5.87,1.06139,5.87,1.95
277,278,2025-06-22,Wapiti_Cr,15,1.21230,87.95,1153.59,1.21647,1.21513,4.17,1.06564,4.17,1.34
297,298,2025-07-07,Wapiti_Cr,15,1.21122,88.25,1020.30,1.21546,1.21391,4.24,0.93205,4.24,1.55
316,317,2025-07-21,Wapiti_Cr,14,1.20592,89.06,1000.35,1.21112,1.20934,5.20,0.91129,5.20,1.78
335,336,2025-08-01,Wapiti_Cr,5,1.22100,89.61,1061.07,1.22200,1.22100,1.00,0.97146,1.00,1.00




--- TSS Data for Site: Warm_Sp ---


,sample_num,date,site,tin_num,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,sed_weight_mg,TSS_conc_mgL-1,susp_solids_mg,susp_sed_mg
0,1,2024-10-11,Warm_Sp,19,1.20800,88.570,1135.00,1.20800,1.20500,0.00,1.046430,0.00,3.00
8,9,2024-10-12,Warm_Sp,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,22,2024-10-19,Warm_Sp,7,1.21700,100.990,1135.94,1.21400,1.21200,-3.00,1.034950,-3.00,2.00
46,47,2024-11-04,Warm_Sp,39,1.22900,87.870,1120.70,1.23600,1.23234,7.00,1.032830,7.00,3.66
54,55,2024-11-09,Warm_Sp,48,1.22200,88.590,1122.91,1.22274,1.22237,0.74,1.034320,0.74,0.37
71,72,2024-11-24,Warm_Sp,14,1.19937,88.320,1138.31,1.24092,1.23873,41.55,1.049990,41.55,2.19
86,87,2025-01-17,Warm_Sp,46,1.23195,87.440,1123.14,1.23693,1.23622,4.98,1.035700,4.98,0.71
101,102,2025-01-30,Warm_Sp,8,1.23845,88.330,1168.10,1.24267,1.24186,4.22,1.079770,4.22,0.81
116,117,2025-02-15,Warm_Sp,50,1.21766,88.710,1123.34,1.22059,1.21977,2.93,1.034630,2.93,0.82
132,133,2025-03-02,Warm_Sp,22,1.23651,88.400,1094.01,1.26331,1.26164,26.80,1.005610,26.80,1.67


In [22]:
# drop the entire site if a TSS column is missing data
tss_df_complete = tss_df.dropna(subset=tss_cols)

In [23]:
# For each site show the number of dates, the number of bottles, and the min sediment weight
# A mismatch between date count and sitecount means that site may have been visited but no bottles collected?
summary = tss_df.groupby("site").agg({
    'date': 'count',
    'site': 'count',
    'susp_sed_mg': 'min'
})
display(summary)

,date,site,susp_sed_mg
site,,,
Avalanche_Cr,27,28,-2.000
BJ_Cr,37,37,0.820
Bear_Cr,20,20,0.000
Camp_Cr,35,35,0.400
Canyon_Cr_Banner,4,4,-0.200
Canyon_Cr_DF2,2,2,1.800
Canyon_Cr_DF4,2,2,1.000
Canyon_Cr_DF5,3,3,0.700
Canyon_Cr_DF8,2,2,1.500


In [24]:
# check for negative values of suspended sediment
negative_tss = tss_df[tss_df['susp_sed_mg'] < 0]
# display records with negative TSS values
negative_tss

,sample_num,date,site,tin_num,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,sed_weight_mg,TSS_conc_mgL-1,susp_solids_mg,susp_sed_mg
278,279,2025-06-22,Flat_Cr,22,1.23853,89.61,1126.14,1.233997,1.23914,-4.533,1.03653,-4.533,-5.143
324,325,2025-08-01,Chapman_Cr,14,1.20800,87.45,1081.21,1.231000,1.23500,23.000,0.99376,23.000,-4.000
326,327,2025-08-01,SFP_Wolf_Cr,22,1.24300,88.51,1140.91,1.244000,1.24700,1.000,1.05240,1.000,-3.000
328,329,2025-08-01,SFP_Canyon_Cr,18,1.24400,88.34,1084.70,1.244000,1.24700,0.000,0.99636,0.000,-3.000
331,332,2025-08-01,HP,7,1.21400,88.57,1140.50,1.215000,1.21800,1.000,1.05193,1.000,-3.000
338,339,2025-08-14,SFP_DW,33,1.23900,88.87,1126.54,1.243000,1.24310,4.000,1.03767,4.000,-0.100
340,341,2025-08-14,SFP_BJ,68,1.22500,88.22,1099.53,1.226000,1.22610,1.000,1.01131,1.000,-0.100
341,342,2025-08-14,Warm_Sp,50,1.21900,89.54,1121.45,1.220000,1.22060,1.000,1.03191,1.000,-0.600
342,343,2025-08-14,SFP_Chap,59,1.22600,88.03,1097.46,1.226000,1.22620,0.000,1.00943,0.000,-0.200
350,351,2025-08-14,HP,29,1.24700,88.07,1138.77,1.250000,1.25030,3.000,1.05070,3.000,-0.300


In [25]:
# see the number of records for each site after dropping missing TSS values
counts = tss_df.groupby("site").size().sort_values(ascending=False)
print(counts)

site
SH_Cr                           38
Canyon_Cr_SFP                   37
BJ_Cr                           37
Warm_Sp                         37
SFP_Canyon_Cr                   36
SFP_BJ                          36
Fox_Cr                          36
SFP_Chap                        36
SFP_Wolf_Cr                     36
SFP_DW                          36
SFP_Alder_Cr                    35
Camp_Cr                         35
Avalanche_Cr                    28
SFP_Wapiti                      27
HP                              27
GO_Cr                           26
Bear_Cr                         20
Wapiti_Cr                       14
Chapman_Cr                       7
Canyon_Cr_Banner                 4
Flat_Cr                          4
Nutcracker_Cr                    3
Canyon_Cr_slump1                 3
Canyon_Cr_DF5                    3
Canyon_Cr_DF2                    2
Canyon_Cr_DF4                    2
Canyon_Cr_DF8                    2
Canyon_cr_avalanche (HP_old)     2
Canyon_Cr_DF_3 

# This section has options to export the cleaned dataset.
Running the cells below will overwrite the existing file.

In [26]:
def export_csv(df: pd.DataFrame, data_name: str, save_path: str) -> None:
    """
    Export a DataFrame to a CSV file in the specified root path.

    Args:
        df (pd.DataFrame): The DataFrame to export.
        data_name (str): The name of the parameter or dataset being exported.
        e.g. 'All', 'TSS', 'DOC', 'ions', 'nutrients'.
        save_path (str): The root path where the CSV file will be saved.

    Returns:
        None
    """
    # get current date
    current_date = datetime.datetime.now().strftime("%Y%m%d")
    if data_name not in {'ALL','TSS', 'DOC', 'IONS', 'NUTRIENTS', 'FIELD_DATA'}:
      print('Invalid data_name. Use ALL, TSS, DOC, IONS, or NUTRIENTS')
      return
    else:
      df_name = f'wapiti_water_{data_name}_sample_data_CLEANED_{current_date}.csv'
      file_path = os.path.join(save_path, df_name)
      df.to_csv(file_path, index=False,na_rep='-9999')
      print(f"DataFrame saved to {file_path}")

In [27]:
# export the full dataset
export_csv(df=df, data_name='FIELD_DATA', save_path=out_path)

DataFrame saved to /content/drive/Shareddrives/RAPID_Wapiti_project/water_samples_QC/wapiti_water_FIELD_DATA_sample_data_CLEANED_20260826.csv


In [28]:
# export the TSS dataset
export_csv(df=tss_df, data_name='TSS', save_path=out_path)

DataFrame saved to /content/drive/Shareddrives/RAPID_Wapiti_project/water_samples_QC/wapiti_water_TSS_sample_data_CLEANED_20260826.csv


In [29]:
# export the Carbon dataset
export_csv(df=toc_df, data_name='DOC', save_path=out_path)

DataFrame saved to /content/drive/Shareddrives/RAPID_Wapiti_project/water_samples_QC/wapiti_water_DOC_sample_data_CLEANED_20260826.csv


In [30]:
# export the combined dataset
export_csv(df=final_df, data_name='ALL', save_path=out_path)

DataFrame saved to /content/drive/Shareddrives/RAPID_Wapiti_project/water_samples_QC/wapiti_water_ALL_sample_data_CLEANED_20260826.csv
